# 5. VAR-CLIP Diagnostic Study: Baseline Quality, Text-Style Control, and Feature Alignment

This notebook deliberately **does not inject VAE features into the transformer**. Its purpose is to answer three questions before we design another reference-image style method:

1. Can frozen VAR-CLIP reliably generate simple, prompt-specified objects?
2. Can it express a visual style when that style is stated directly in text?
3. Does a generated text-style image look closer to a matching style reference in CLIP and VAE-SVD feature space than the photo-prompt image?

```text
20 simple prompts x 10 seeds
        -> baseline reliability audit

same object + five text renderings
        -> prompt-style controllability audit

photo generation / text-style generation / style-reference image
        -> CLIP and VAE-SVD diagnostic comparison
```

The VAE is used only as a frozen **analysis encoder** in Section 3. No VAE latent/token mixing occurs in this notebook.

## What Each Result Means

| Observation | Meaning | Decision |
| --- | --- | --- |
| Baseline prompts often miss the requested object | VAR-CLIP is not reliable enough for subtle style-transfer claims | verify the official checkpoint/sampler or change backbone |
| Text style changes the rendering while preserving the object | VAR-CLIP has native style capacity | build a reference-image-to-native-style bridge |
| Text style does not change rendering | image-reference injection cannot rescue the model | do not continue PFB experiments on this checkpoint |
| Text-style output moves toward the reference in feature statistics | VAE-SVD diagnostics carry useful style evidence | use them only as secondary analysis, not as the objective |
| Text-style output stays far from reference | style is mainly prompt-level/category-level, not personalized | test prototype retrieval or a lightweight adapter |

The SVD statistics are **diagnostic**, not a proof of style similarity. VAE features retain content information, especially at coarse scales.

In [ ]:
# Colab setup: Runtime -> Change runtime type -> T4 GPU (or better), then run this cell.
!nvidia-smi

import os
import subprocess
from pathlib import Path

assert os.path.exists('/usr/local/cuda') or os.environ.get('COLAB_GPU'), (
    'No GPU runtime detected. In Colab, select Runtime -> Change runtime type -> T4 GPU, then reconnect.'
)

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

VAR_CLIP_REPO = 'https://github.com/daixiangzi/VAR-CLIP.git'
VAR_CLIP_DIR = RUNTIME_ROOT / 'VAR-CLIP'
OUTPUT_DIR = RUNTIME_ROOT / 'VAR_CLIP_outputs' / 'diagnostic_study'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VAR_CLIP_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', VAR_CLIP_REPO, str(VAR_CLIP_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'reset', '--hard', 'origin/master'], check=True)

os.chdir(VAR_CLIP_DIR)
print('VAR-CLIP source:', VAR_CLIP_DIR)
print('Results:', OUTPUT_DIR)

In [ ]:
# Keep Colab's CUDA-enabled PyTorch. This package provides the `open_clip` import used upstream.
!pip -q install gdown huggingface_hub einops typed-argument-parser pytz open_clip_torch pandas tqdm seaborn

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'A CUDA GPU is required.'

from huggingface_hub import hf_hub_download
import gdown

PRETRAINED_DIR = VAR_CLIP_DIR / 'pretrained'
LOCAL_OUTPUT_DIR = VAR_CLIP_DIR / 'local_output'
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vae_path = Path(hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=PRETRAINED_DIR,
))

clip_path = PRETRAINED_DIR / 'ViT-L-14.pt'
OPENAI_CLIP_VIT_L14_URL = (
    'https://openaipublic.azureedge.net/clip/models/'
    'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt'
)
if not clip_path.exists() or clip_path.stat().st_size < 500_000_000:
    subprocess.run(['wget', '-c', '--show-progress', '-O', str(clip_path), OPENAI_CLIP_VIT_L14_URL], check=True)

var_clip_path = LOCAL_OUTPUT_DIR / 'ar-ckpt-last.pth'
VAR_CLIP_CHECKPOINT_URL = 'https://drive.google.com/file/d/10gSxvaKaNKJcnqFhU7hQywU28w3nbgoV/view?usp=sharing'
if not var_clip_path.exists() or var_clip_path.stat().st_size < 100_000_000:
    gdown.download(url=VAR_CLIP_CHECKPOINT_URL, output=str(var_clip_path), fuzzy=True)

assert vae_path.exists(), vae_path
assert clip_path.exists() and clip_path.stat().st_size > 500_000_000, 'Incomplete OpenAI CLIP download.'
assert var_clip_path.exists() and var_clip_path.stat().st_size > 100_000_000, 'Incomplete VAR-CLIP download.'
print('All checkpoints are available.')


In [ ]:
import gc
import math
import random
import sys
import importlib
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# PyTorch 2.6+ compatibility for the trusted OpenAI TorchScript checkpoint expected by VAR-CLIP.
clip_source = VAR_CLIP_DIR / 'models' / 'clip.py'
clip_source_text = clip_source.read_text()
old_clip_call = "pretrained='pretrained/ViT-L-14.pt')"
new_clip_call = "pretrained='pretrained/ViT-L-14.pt', weights_only=False)"
if old_clip_call in clip_source_text:
    clip_source.write_text(clip_source_text.replace(old_clip_call, new_clip_call, 1))
assert 'weights_only=False' in clip_source.read_text()
if 'models.clip' in sys.modules:
    importlib.reload(sys.modules['models.clip'])

# Avoid unnecessary default initialization before loading the checkpoints.
setattr(torch.nn.Linear, 'reset_parameters', lambda self: None)
setattr(torch.nn.LayerNorm, 'reset_parameters', lambda self: None)

from clip_util import CLIPWrapper
from models.clip import clip_vit_l14
from tokenizer import tokenize
from models import build_vae_var
from models.helpers import sample_with_top_k_top_p_

MODEL_DEPTH = 16
PATCH_NUMS = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda'

vae, var_clip = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=PATCH_NUMS,
    n_cond_embed=768,
    depth=MODEL_DEPTH,
    shared_aln=False,
)
clip_model = CLIPWrapper(clip_vit_l14(pretrained=True).to(device).eval(), normalize=True)
vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
checkpoint = torch.load(var_clip_path, map_location='cpu')
var_clip.load_state_dict(checkpoint['trainer']['var_wo_ddp'], strict=True)
vae.eval()
var_clip.eval()
for model in (vae, var_clip):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print('Frozen VAR-CLIP-d16 and CLIP ViT-L/14 are ready.')


## Generation Helpers

This is ordinary VAR-CLIP autoregressive inference. It has no PFB, SAC, style-reference feature injection, VAE token replacement, or dual generation stream.

A seed is fixed per sample so that style variants can be compared under the same sampling noise.

In [ ]:
def prepare_prompt_embedding(prompt):
    tokens = tokenize([prompt]).to(device)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        return clip_model.encode_text(tokens)


def _disable_cache(model):
    for block in model.blocks:
        block.attn.kv_caching(False)


def _prompt_condition(model, prompt_embedding):
    batch = prompt_embedding.shape[0]
    null_embedding = model.noise(torch.tensor(0, device=prompt_embedding.device)).unsqueeze(0).expand(batch, -1)
    return model.cond_proj(torch.cat((prompt_embedding, null_embedding), dim=0))


def _next_tokens_from_fhat(model, feature, next_step, level_position, current_length):
    patch_num = model.patch_nums[next_step]
    tokens = F.interpolate(feature, size=(patch_num, patch_num), mode='area')
    tokens = tokens.view(tokens.shape[0], model.Cvae, -1).transpose(1, 2)
    tokens = model.word_embed(tokens) + level_position[:, current_length:current_length + patch_num * patch_num]
    return tokens.repeat(2, 1, 1)  # conditional and classifier-free null branches


@torch.no_grad()
def generate_var_clip(model, prompt_embedding, *, seed=0, cfg=4.0, top_k=900, top_p=0.95, return_trace=False):
    """Single-stream, unmodified VAR-CLIP AR generation."""
    batch = prompt_embedding.shape[0]
    if batch != 1:
        raise ValueError('This notebook currently uses B=1 generation for reproducible per-seed analysis.')

    model.eval()
    rng = torch.Generator(device=prompt_embedding.device).manual_seed(int(seed))
    with torch.autocast('cuda', dtype=torch.float16):
        condition = _prompt_condition(model, prompt_embedding)
        level_position = model.lvl_embed(model.lvl_1L) + model.pos_1LC
        tokens = (
            condition.unsqueeze(1).expand(2 * batch, model.first_l, -1)
            + model.pos_start.expand(2 * batch, model.first_l, -1)
            + level_position[:, :model.first_l]
        )
        fhat = condition.new_zeros(batch, model.Cvae, model.patch_nums[-1], model.patch_nums[-1])
        trace = []

        _disable_cache(model)
        for block in model.blocks:
            block.attn.kv_caching(True)

        try:
            current_length = 0
            for step_id, patch_num in enumerate(model.patch_nums):
                current_length += patch_num * patch_num
                condition_for_blocks = model.shared_ada_lin(condition)
                hidden = tokens
                for block in model.blocks:
                    hidden = block(x=hidden, cond_BD=condition_for_blocks, attn_bias=None)

                logits = model.get_logits(hidden, condition)
                cfg_ratio = cfg * (step_id / model.num_stages_minus_1)
                logits = (1 + cfg_ratio) * logits[:batch] - cfg_ratio * logits[batch:]
                indices = sample_with_top_k_top_p_(
                    logits, rng=rng, top_k=top_k, top_p=top_p, num_samples=1
                )[:, :, 0]
                residual = model.vae_quant_proxy[0].embedding(indices).transpose(1, 2).reshape(
                    batch, model.Cvae, patch_num, patch_num
                )
                fhat, _ = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), fhat, residual
                )
                if return_trace:
                    trace.append(fhat.detach().float().clone())

                if step_id != model.num_stages_minus_1:
                    tokens = _next_tokens_from_fhat(model, fhat, step_id + 1, level_position, current_length)

            image_01 = model.vae_proxy[0].fhat_to_img(fhat).add(1).mul(0.5).clamp(0, 1)
            return {'image_01': image_01, 'fhat_scales': trace}
        finally:
            _disable_cache(model)


def slugify(text):
    return re.sub(r'[^a-z0-9]+', '_', text.lower()).strip('_')


def save_image_01(image_01, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torchvision.utils.save_image(image_01.detach().float().cpu().clamp(0, 1), path)


def read_image_01(path):
    image = Image.open(path).convert('RGB')
    return torchvision.transforms.functional.to_tensor(image).unsqueeze(0)


def show_grid(items, title, ncols=5, image_size=3.0):
    rows = math.ceil(len(items) / ncols)
    plt.figure(figsize=(image_size * ncols, image_size * rows))
    for index, (name, image) in enumerate(items):
        plt.subplot(rows, ncols, index + 1)
        if isinstance(image, (str, Path)):
            image = read_image_01(image)
        image = image.detach().float().cpu()[0]
        plt.imshow(image.clamp(0, 1).permute(1, 2, 0).numpy())
        plt.title(name, fontsize=9)
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## 1. Baseline Reliability: 20 Objects x 10 Images

This is the foundational audit. It creates **200 ordinary, prompt-only VAR-CLIP images** and saves a 10-image grid for each object. The run is resumable: rerunning the cell skips images already saved.

Use these grids to annotate two simple human scores per prompt:

- **object fidelity:** does the result contain the requested object?
- **image quality:** is it coherent enough that a small style change could be judged?

Do not use a difficult prompt as evidence against style control. A style method cannot recover an object the base generator never formed.

In [ ]:
BASELINE_PROMPTS = [
    ('golden_retriever', 'a photo of a golden retriever sitting on grass'),
    ('tabby_cat', 'a photo of a tabby cat sitting indoors'),
    ('african_elephant', 'a photo of an african elephant standing in a field'),
    ('zebra', 'a photo of a zebra standing on grass'),
    ('red_fox', 'a photo of a red fox in a forest'),
    ('sports_car', 'a photo of a red sports car parked on a street'),
    ('school_bus', 'a photo of a yellow school bus on a road'),
    ('steam_train', 'a photo of a steam train on railway tracks'),
    ('sailing_ship', 'a photo of a sailing ship on the ocean'),
    ('mountain_bike', 'a photo of a mountain bike on a trail'),
    ('pineapple', 'a photo of a pineapple on a table'),
    ('red_apple', 'a photo of a red apple on a table'),
    ('sunflower', 'a photo of a sunflower in a field'),
    ('mushroom', 'a photo of a mushroom in a forest'),
    ('acoustic_guitar', 'a photo of an acoustic guitar'),
    ('coffee_mug', 'a photo of a coffee mug on a table'),
    ('teapot', 'a photo of a teapot on a table'),
    ('alarm_clock', 'a photo of an alarm clock on a table'),
    ('lighthouse', 'a photo of a lighthouse by the ocean'),
    ('snowy_mountain', 'a photo of a snowy mountain landscape'),
]

N_BASELINE_SAMPLES = 10
BASELINE_SEEDS = tuple(range(N_BASELINE_SAMPLES))
BASELINE_CFG = 4.0
BASELINE_TOP_K = 900
BASELINE_TOP_P = 0.95
RUN_BASELINE_SUITE = True

BASELINE_DIR = OUTPUT_DIR / 'baseline_20_objects'
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

if RUN_BASELINE_SUITE:
    records = []
    total = len(BASELINE_PROMPTS) * len(BASELINE_SEEDS)
    progress = tqdm(total=total, desc='Baseline samples')
    for object_id, prompt in BASELINE_PROMPTS:
        prompt_embedding = prepare_prompt_embedding(prompt)
        object_dir = BASELINE_DIR / object_id
        object_dir.mkdir(parents=True, exist_ok=True)
        for seed in BASELINE_SEEDS:
            out_path = object_dir / f'seed_{seed:02d}.png'
            if not out_path.exists():
                output = generate_var_clip(
                    var_clip, prompt_embedding, seed=seed, cfg=BASELINE_CFG,
                    top_k=BASELINE_TOP_K, top_p=BASELINE_TOP_P,
                )
                save_image_01(output['image_01'], out_path)
                del output
                torch.cuda.empty_cache()
            records.append({'object_id': object_id, 'prompt': prompt, 'seed': seed, 'image_path': str(out_path)})
            progress.update(1)
        grid_path = BASELINE_DIR / f'{object_id}_grid.png'
        if not grid_path.exists():
            grid_images = torch.cat([read_image_01(object_dir / f'seed_{seed:02d}.png') for seed in BASELINE_SEEDS], dim=0)
            grid = torchvision.utils.make_grid(grid_images, nrow=N_BASELINE_SAMPLES, padding=2, pad_value=1.0)
            Image.fromarray(grid.permute(1, 2, 0).mul(255).byte().numpy()).save(grid_path)
    progress.close()

    baseline_manifest = pd.DataFrame(records)
    baseline_manifest.to_csv(BASELINE_DIR / 'manifest.csv', index=False)
    preview = [(object_id, BASELINE_DIR / object_id / 'seed_00.png') for object_id, _ in BASELINE_PROMPTS]
    show_grid(preview, 'Baseline preview: seed 0 for all 20 prompts', ncols=5)
    print('Saved 20 ten-sample grids in:', BASELINE_DIR)


## 2. Does VAR-CLIP Understand Style From Text?

This is the first large sweep. We keep the object prompt stable, change only the style words, and repeat the same seeds for every style.

Default size:

```text
20 object prompts x 8 style variants x 6 seeds = 960 images
```

This is intentionally large because one prompt can lie. We need repeated objects, repeated styles, and repeated seeds before trusting any conclusion about VAR-CLIP style control.

The cell is resumable: every generated image is saved immediately. If Colab/Kaggle disconnects, rerun the cell and it skips images already on disk.


In [ ]:

STYLE_OBJECTS = [
    ('golden_retriever', 'a golden retriever sitting on grass'),
    ('tabby_cat', 'a tabby cat sitting indoors'),
    ('african_elephant', 'an african elephant standing in a field'),
    ('zebra', 'a zebra standing on grass'),
    ('red_fox', 'a red fox in a forest'),
    ('sports_car', 'a red sports car parked on a street'),
    ('school_bus', 'a yellow school bus on a road'),
    ('steam_train', 'a steam train on railway tracks'),
    ('sailing_ship', 'a sailing ship on the ocean'),
    ('mountain_bike', 'a mountain bike on a trail'),
    ('pineapple', 'a pineapple on a table'),
    ('red_apple', 'a red apple on a table'),
    ('sunflower', 'a sunflower in a field'),
    ('mushroom', 'a mushroom in a forest'),
    ('acoustic_guitar', 'an acoustic guitar'),
    ('coffee_mug', 'a coffee mug on a table'),
    ('teapot', 'a teapot on a table'),
    ('alarm_clock', 'an alarm clock on a table'),
    ('lighthouse', 'a lighthouse by the ocean'),
    ('snowy_mountain', 'a snowy mountain landscape'),
]

STYLE_VARIANTS = [
    ('photo', 'a photo of {subject}'),
    ('pencil_sketch', 'a detailed pencil sketch of {subject}'),
    ('line_drawing', 'a clean black-and-white line drawing of {subject}'),
    ('watercolor', 'a soft watercolor painting of {subject}'),
    ('oil_painting', 'an oil painting of {subject}'),
    ('pixel_art', 'a pixel art image of {subject}'),
    ('anime_style', 'an anime illustration of {subject}'),
    ('cyberpunk', 'a cyberpunk neon illustration of {subject}'),
]

N_STYLE_SEEDS = 6
STYLE_SEEDS = tuple(range(N_STYLE_SEEDS))
RUN_TEXT_STYLE_SUITE = True
STYLE_TEXT_DIR = OUTPUT_DIR / 'text_style_control_large_sweep'
STYLE_TEXT_DIR.mkdir(parents=True, exist_ok=True)

if RUN_TEXT_STYLE_SUITE:
    records = []
    total = len(STYLE_OBJECTS) * len(STYLE_VARIANTS) * len(STYLE_SEEDS)
    progress = tqdm(total=total, desc='Text-style control large sweep')
    for object_id, subject in STYLE_OBJECTS:
        object_dir = STYLE_TEXT_DIR / object_id
        object_dir.mkdir(parents=True, exist_ok=True)
        for style_id, template in STYLE_VARIANTS:
            prompt = template.format(subject=subject)
            prompt_embedding = prepare_prompt_embedding(prompt)
            for seed in STYLE_SEEDS:
                out_path = object_dir / f'{style_id}_seed_{seed:02d}.png'
                if not out_path.exists():
                    output = generate_var_clip(
                        var_clip, prompt_embedding, seed=seed, cfg=BASELINE_CFG,
                        top_k=BASELINE_TOP_K, top_p=BASELINE_TOP_P,
                    )
                    save_image_01(output['image_01'], out_path)
                    del output
                    torch.cuda.empty_cache()
                records.append({
                    'object_id': object_id, 'subject': subject, 'style_id': style_id,
                    'prompt': prompt, 'seed': seed, 'image_path': str(out_path),
                })
                progress.update(1)

        # Each row is one seed; each column changes only the style phrase.
        grid_items = []
        for seed in STYLE_SEEDS:
            for style_id, _ in STYLE_VARIANTS:
                grid_items.append(read_image_01(object_dir / f'{style_id}_seed_{seed:02d}.png'))
        grid = torchvision.utils.make_grid(
            torch.cat(grid_items, dim=0), nrow=len(STYLE_VARIANTS), padding=2, pad_value=1.0
        )
        Image.fromarray(grid.permute(1, 2, 0).mul(255).byte().numpy()).save(object_dir / 'style_prompt_grid.png')
    progress.close()

    style_manifest = pd.DataFrame(records)
    style_manifest.to_csv(STYLE_TEXT_DIR / 'manifest.csv', index=False)
    preview = []
    for object_id, _ in STYLE_OBJECTS[:10]:
        preview.append((object_id, STYLE_TEXT_DIR / object_id / 'pencil_sketch_seed_00.png'))
    show_grid(preview, 'Large text-style sweep preview: pencil sketch, seed 0', ncols=5)
    print('Saved large text-style grids in:', STYLE_TEXT_DIR)


## 3. Feature-Space Comparison: Photo vs Text Style vs Reference Style

We compare three images for one controlled subject:

```text
A: photo-prompt generation
B: same subject, pencil-sketch-prompt generation
R: pencil-sketch reference image
```

For every VAE scale, we measure:

- **CLIP cosine**: global image similarity. It is semantically confounded, so interpret cautiously.
- **SVD spectrum cosine**: whether the relative singular-value profile is similar.
- **principal-subspace similarity**: channel-direction overlap of the top SVD components.
- **Phi-feature cosine**: similarity after the paper-style exponentially weighted SVD extractor.

The most useful evidence is whether `B -> R` is consistently closer than `A -> R`. This does not prove style transfer; it only tests whether native text styling moves the image toward the reference in these frozen representations.

In [ ]:
# We use a same-subject dog sketch reference to reduce semantic mismatch in the first comparison.
STYLE_WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
STYLE_WORKSPACE = RUNTIME_ROOT / 'VAR_Style_Transfer_Workspace'
if not STYLE_WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', STYLE_WORKSPACE_REPO, str(STYLE_WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'reset', '--hard', 'origin/main'], check=True)

ANALYSIS_OBJECT_ID = 'golden_retriever'
ANALYSIS_SUBJECT = 'a golden retriever sitting on grass'
ANALYSIS_SEED = 0
ANALYSIS_PHOTO_PROMPT = 'a photo of ' + ANALYSIS_SUBJECT
ANALYSIS_STYLE_PROMPT = 'a detailed pencil sketch of ' + ANALYSIS_SUBJECT
ANALYSIS_REFERENCE_PATH = STYLE_WORKSPACE / 'style' / 'Sketch' / 'S005.png'
assert ANALYSIS_REFERENCE_PATH.exists(), ANALYSIS_REFERENCE_PATH
ANALYSIS_DIR = OUTPUT_DIR / 'feature_alignment'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)


def ensure_generated(prompt, name):
    path = ANALYSIS_DIR / f'{name}_seed_{ANALYSIS_SEED:02d}.png'
    if not path.exists():
        output = generate_var_clip(
            var_clip, prepare_prompt_embedding(prompt), seed=ANALYSIS_SEED,
            cfg=BASELINE_CFG, top_k=BASELINE_TOP_K, top_p=BASELINE_TOP_P,
        )
        save_image_01(output['image_01'], path)
        del output
        torch.cuda.empty_cache()
    return path

photo_path = ensure_generated(ANALYSIS_PHOTO_PROMPT, 'photo')
stylized_path = ensure_generated(ANALYSIS_STYLE_PROMPT, 'pencil_sketch')
reference_image = ImageOps.fit(Image.open(ANALYSIS_REFERENCE_PATH).convert('RGB'), (256, 256), method=Image.Resampling.LANCZOS)
reference_path = ANALYSIS_DIR / 'reference_pencil_sketch.png'
reference_image.save(reference_path)

show_grid(
    [('A: photo prompt', photo_path), ('B: pencil-sketch prompt', stylized_path), ('R: style reference', reference_path)],
    'Controlled images for feature comparison', ncols=3, image_size=4,
)

In [ ]:
CLIP_MEAN = torch.tensor((0.48145466, 0.4578275, 0.40821073), device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor((0.26862954, 0.26130258, 0.27577711), device=device).view(1, 3, 1, 1)


def image_path_to_01(path, size=256):
    image = ImageOps.fit(Image.open(path).convert('RGB'), (size, size), method=Image.Resampling.LANCZOS)
    return torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)


@torch.no_grad()
def clip_image_embedding(image_01):
    image = F.interpolate(image_01, size=(224, 224), mode='bicubic', align_corners=False)
    image = (image - CLIP_MEAN) / CLIP_STD
    return clip_model.encode_image(image)


@torch.no_grad()
def image_to_fhat_scales(image_01):
    image_m11 = image_01.mul(2).sub(1)
    latent = vae.quant_conv(vae.encoder(image_m11))
    return [feature.float() for feature in vae.quantize.f_to_idxBl_or_fhat(latent, to_fhat=True)]


def feature_matrix(feature_bchw, center=True):
    matrix = feature_bchw[0].permute(1, 2, 0).reshape(-1, feature_bchw.shape[1]).float()
    return matrix - matrix.mean(dim=0, keepdim=True) if center else matrix


def svd_signature(feature_bchw, rank=8, alpha=1.0):
    matrix = feature_matrix(feature_bchw, center=True)
    u, singular_values, vh = torch.linalg.svd(matrix, full_matrices=False)
    used_rank = min(rank, singular_values.numel())
    spectrum = singular_values / singular_values.sum().clamp_min(1e-8)
    weights = torch.exp(-alpha * torch.arange(used_rank, device=matrix.device, dtype=matrix.dtype))
    phi_matrix = (u[:, :used_rank] * (singular_values[:used_rank] * weights).unsqueeze(0)) @ vh[:used_rank]
    return {
        'spectrum': spectrum,
        'basis': vh[:used_rank].T,  # C x rank channel subspace
        'phi': phi_matrix.flatten(),
        'singular_values': singular_values,
    }


def cosine(x, y):
    return F.cosine_similarity(x.flatten().unsqueeze(0), y.flatten().unsqueeze(0)).item()


def subspace_similarity(basis_a, basis_b):
    # Mean cosine of principal angles between the two top-rank channel subspaces.
    return torch.linalg.svdvals(basis_a.T @ basis_b).mean().item()


def compare_signatures(left, right):
    return {
        'spectrum_cosine': cosine(left['spectrum'], right['spectrum']),
        'subspace_similarity': subspace_similarity(left['basis'], right['basis']),
        'phi_cosine': cosine(left['phi'], right['phi']),
    }


photo_01 = image_path_to_01(photo_path)
stylized_01 = image_path_to_01(stylized_path)
reference_01 = image_path_to_01(reference_path)

with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    clip_photo = clip_image_embedding(photo_01)
    clip_stylized = clip_image_embedding(stylized_01)
    clip_reference = clip_image_embedding(reference_01)

photo_fhat = image_to_fhat_scales(photo_01)
stylized_fhat = image_to_fhat_scales(stylized_01)
reference_fhat = image_to_fhat_scales(reference_01)
print('VAE feature shapes:', [tuple(feature.shape) for feature in photo_fhat])

In [ ]:
SVD_RANK = 8
SVD_ALPHA = 1.0
analysis_rows = []

for scale_id, (photo_feature, stylized_feature, reference_feature) in enumerate(
    zip(photo_fhat, stylized_fhat, reference_fhat)
):
    photo_sig = svd_signature(photo_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    stylized_sig = svd_signature(stylized_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    reference_sig = svd_signature(reference_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    photo_to_reference = compare_signatures(photo_sig, reference_sig)
    stylized_to_reference = compare_signatures(stylized_sig, reference_sig)
    delta_norm = (stylized_feature - photo_feature).norm().item() / photo_feature.norm().clamp_min(1e-8).item()
    analysis_rows.append({
        'scale_id': scale_id,
        'resolution': f'{PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}',
        'relative_feature_change_B_minus_A': delta_norm,
        'photo_reference_spectrum_cosine': photo_to_reference['spectrum_cosine'],
        'text_style_reference_spectrum_cosine': stylized_to_reference['spectrum_cosine'],
        'spectrum_gain_B_minus_A': stylized_to_reference['spectrum_cosine'] - photo_to_reference['spectrum_cosine'],
        'photo_reference_subspace_similarity': photo_to_reference['subspace_similarity'],
        'text_style_reference_subspace_similarity': stylized_to_reference['subspace_similarity'],
        'subspace_gain_B_minus_A': stylized_to_reference['subspace_similarity'] - photo_to_reference['subspace_similarity'],
        'photo_reference_phi_cosine': photo_to_reference['phi_cosine'],
        'text_style_reference_phi_cosine': stylized_to_reference['phi_cosine'],
        'phi_gain_B_minus_A': stylized_to_reference['phi_cosine'] - photo_to_reference['phi_cosine'],
    })

analysis_table = pd.DataFrame(analysis_rows)
analysis_table.to_csv(ANALYSIS_DIR / 'vae_svd_comparison.csv', index=False)

clip_rows = pd.DataFrame([{
    'CLIP_cosine(photo, reference)': cosine(clip_photo, clip_reference),
    'CLIP_cosine(text_style, reference)': cosine(clip_stylized, clip_reference),
    'CLIP_cosine(photo, text_style)': cosine(clip_photo, clip_stylized),
}])
clip_rows.to_csv(ANALYSIS_DIR / 'clip_comparison.csv', index=False)

print('CLIP global-image comparison')
display(clip_rows.round(4))
print('VAE-SVD comparison. Positive gain means the text-style image moved closer to the reference at that scale.')
display(analysis_table.round(4))

In [ ]:
# Visualize the actual difference B - A and the SVD spectra of A, B, and R.
selected_scales = [2, 5, 9]  # 3x3, 6x6, 16x16 in VAR's ten-scale pyramid
plt.figure(figsize=(15, 8))
for column, scale_id in enumerate(selected_scales):
    difference_map = (stylized_fhat[scale_id] - photo_fhat[scale_id]).square().sum(dim=1).sqrt()[0].cpu()
    plt.subplot(2, len(selected_scales), column + 1)
    plt.imshow(difference_map, cmap='magma')
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(f'Feature-change map B-A\nscale {scale_id}: {PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}')
    plt.axis('off')

    plt.subplot(2, len(selected_scales), len(selected_scales) + column + 1)
    for label, feature, color in [
        ('A: photo', photo_fhat[scale_id], '#4c78a8'),
        ('B: text style', stylized_fhat[scale_id], '#f58518'),
        ('R: reference', reference_fhat[scale_id], '#54a24b'),
    ]:
        signature = svd_signature(feature, rank=SVD_RANK, alpha=SVD_ALPHA)
        spectrum = signature['spectrum'].detach().cpu().numpy()
        plt.plot(np.arange(1, len(spectrum) + 1), spectrum, marker='o', ms=3, label=label, color=color)
    plt.title(f'Normalized singular spectrum\nscale {scale_id}')
    plt.xlabel('component rank')
    plt.ylabel('relative singular value')
    plt.legend(fontsize=8)
    plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'feature_difference_and_spectra.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved diagnostics in:', ANALYSIS_DIR)

### 3.1 Direct Test: Does the Text-Style Change Point Toward the Reference Style Feature?

The preceding table compares whole images. This extra diagnostic directly compares:

```text
D_raw[s] = F_text_style[s] - F_photo[s]
D_phi[s] = Phi(F_text_style[s]) - Phi(F_photo[s])
S_ref[s] = Phi(F_reference[s])
```

`D_raw` is the VAE-feature change caused by adding the text style phrase. `D_phi` is the same change after principal-feature extraction. `S_ref` is the reference image's principal feature.

At each scale we measure cosine alignment, SVD spectrum alignment, channel-subspace alignment, and a spatially agnostic Gram/covariance alignment between the generated style change and the reference style feature.

This remains a **diagnostic**, not a claim that `S_ref` is perfectly content-free: the style reference still contains its own dog drawing. The same-subject dog reference deliberately reduces, but cannot eliminate, that confound.

In [ ]:
def feature_gram(feature_bchw):
    matrix = feature_matrix(feature_bchw, center=True)
    gram = matrix.T @ matrix / max(matrix.shape[0] - 1, 1)
    return gram / gram.norm().clamp_min(1e-8)


direction_rows = []
for scale_id, (photo_feature, stylized_feature, reference_feature) in enumerate(
    zip(photo_fhat, stylized_fhat, reference_fhat)
):
    delta_raw = stylized_feature - photo_feature

    photo_sig = svd_signature(photo_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    stylized_sig = svd_signature(stylized_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    reference_sig = svd_signature(reference_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    delta_sig = svd_signature(delta_raw, rank=SVD_RANK, alpha=SVD_ALPHA)

    # Two forms of the generated style direction, compared with the reference principal feature.
    delta_phi = stylized_sig['phi'] - photo_sig['phi']
    reference_phi = reference_sig['phi']
    raw_to_reference_phi = cosine(delta_raw.flatten(), reference_phi)
    phi_delta_to_reference_phi = cosine(delta_phi, reference_phi)
    gram_alignment = cosine(feature_gram(delta_raw), feature_gram(reference_feature))

    direction_rows.append({
        'scale_id': scale_id,
        'resolution': f'{PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}',
        'raw_delta_to_reference_phi_cosine': raw_to_reference_phi,
        'principal_delta_to_reference_phi_cosine': phi_delta_to_reference_phi,
        'raw_delta_reference_spectrum_cosine': cosine(delta_sig['spectrum'], reference_sig['spectrum']),
        'raw_delta_reference_subspace_similarity': subspace_similarity(delta_sig['basis'], reference_sig['basis']),
        'raw_delta_reference_gram_cosine': gram_alignment,
    })

direction_table = pd.DataFrame(direction_rows)
direction_table.to_csv(ANALYSIS_DIR / 'text_style_direction_vs_reference.csv', index=False)
print('Direct style-direction vs reference-feature comparison')
display(direction_table.round(4))

selected_scales = [2, 5, 9]  # 3x3, 6x6, 16x16
fig, axes = plt.subplots(3, len(selected_scales), figsize=(15, 11))
for column, scale_id in enumerate(selected_scales):
    photo_feature = photo_fhat[scale_id]
    stylized_feature = stylized_fhat[scale_id]
    reference_feature = reference_fhat[scale_id]
    delta_raw = stylized_feature - photo_feature

    photo_sig = svd_signature(photo_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    stylized_sig = svd_signature(stylized_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    reference_sig = svd_signature(reference_feature, rank=SVD_RANK, alpha=SVD_ALPHA)
    delta_sig = svd_signature(delta_raw, rank=SVD_RANK, alpha=SVD_ALPHA)

    # Where did the text style phrase actually change the image feature?
    change_map = delta_raw.square().sum(dim=1).sqrt()[0].cpu()
    axis = axes[0, column]
    im = axis.imshow(change_map, cmap='magma')
    axis.set_title(f'|D_raw| at scale {scale_id}\n{PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}')
    axis.axis('off')
    fig.colorbar(im, ax=axis, fraction=0.046, pad=0.04)

    # Does the energy distribution of the generated direction resemble the reference feature?
    axis = axes[1, column]
    axis.plot(delta_sig['spectrum'].detach().cpu().numpy(), marker='o', ms=3, label='D_raw: B - A', color='#f58518')
    axis.plot(reference_sig['spectrum'].detach().cpu().numpy(), marker='o', ms=3, label='S_ref', color='#54a24b')
    axis.set_title(f'SVD spectra: D_raw vs S_ref\nscale {scale_id}')
    axis.set_xlabel('component rank')
    axis.set_ylabel('relative singular value')
    axis.grid(alpha=0.25)
    if column == 0:
        axis.legend(fontsize=8)

    # Direct alignment metrics: higher is more similar in the frozen VAE representation.
    axis = axes[2, column]
    metric_row = direction_table.loc[direction_table.scale_id == scale_id].iloc[0]
    names = ['raw delta\nvs Phi(ref)', 'Phi delta\nvs Phi(ref)', 'Gram\nalignment']
    values = [
        metric_row['raw_delta_to_reference_phi_cosine'],
        metric_row['principal_delta_to_reference_phi_cosine'],
        metric_row['raw_delta_reference_gram_cosine'],
    ]
    bars = axis.bar(names, values, color=['#4c78a8', '#f58518', '#54a24b'])
    axis.set_ylim(-1, 1)
    axis.axhline(0, color='black', linewidth=0.8)
    axis.set_title(f'Direction-reference alignment\nscale {scale_id}')
    axis.tick_params(axis='x', labelsize=8)
    for bar, value in zip(bars, values):
        axis.text(bar.get_x() + bar.get_width() / 2, value + (0.03 if value >= 0 else -0.07), f'{value:.2f}', ha='center', fontsize=8)

plt.suptitle('Does the text-style feature direction align with the reference style feature?', y=1.02)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'text_style_direction_vs_reference.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved direct direction analysis in:', ANALYSIS_DIR)


### 3.2 Large All-Scale Pattern Sweep

Now we stop looking at one dog example and measure the whole bank from Section 2.

For every generated pair:

```text
photo prompt image A(o, seed)
style prompt image B(o, style, seed)
style direction D(o, style, seed, scale) = normalize(B_feature - A_feature)
```

Then we ask three questions at every VAR scale:

```text
1. Is the same style direction consistent across different objects and seeds?
2. Are different style directions separated from each other?
3. Does the text-style direction point toward real style-reference images from the style folder?
```

This is the key diagnostic for your research direction. If this section is weak, then reference style injection is not failing because our alpha is wrong; it is failing because the representation itself does not give us a clean style handle.


In [ ]:

RUN_ALL_SCALE_STYLE_DIRECTION_SWEEP = True
FEATURE_SWEEP_DIR = OUTPUT_DIR / 'all_scale_style_direction_sweep'
FEATURE_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR = FEATURE_SWEEP_DIR / 'vae_fhat_cache'
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SWEEP_STYLE_IDS = [style_id for style_id, _ in STYLE_VARIANTS if style_id != 'photo']
FEATURE_SWEEP_OBJECT_IDS = [object_id for object_id, _ in STYLE_OBJECTS]
FEATURE_SWEEP_SEEDS = STYLE_SEEDS
FEATURE_SWEEP_RANK = 8
FEATURE_SWEEP_ALPHA = 1.0
MAX_PAIRWISE_COSINE_PAIRS = 3000

STYLE_REFERENCE_FILES = {
    'pencil_sketch': [
        STYLE_WORKSPACE / 'style' / 'Sketch' / 'S005.png',
        STYLE_WORKSPACE / 'style' / 'Sketch' / 'S001.png',
        STYLE_WORKSPACE / 'style' / 'Sketch' / 'S010.png',
    ],
    'line_drawing': [
        STYLE_WORKSPACE / 'style' / 'LineDrawing' / 'LD001.png',
        STYLE_WORKSPACE / 'style' / 'LineDrawing' / 'LD002.png',
        STYLE_WORKSPACE / 'style' / 'LineDrawing' / 'LD006.png',
    ],
    'watercolor': [
        STYLE_WORKSPACE / 'style' / 'WaterColor' / 'WC001.png',
        STYLE_WORKSPACE / 'style' / 'WaterColor' / 'WC002.png',
        STYLE_WORKSPACE / 'style' / 'WaterColor' / 'WC004.png',
    ],
    'oil_painting': [
        STYLE_WORKSPACE / 'style' / 'OilPainting' / 'OP001.png',
        STYLE_WORKSPACE / 'style' / 'OilPainting' / 'OP005.png',
        STYLE_WORKSPACE / 'style' / 'OilPainting' / 'OP006.png',
    ],
    'pixel_art': [
        STYLE_WORKSPACE / 'style' / 'PixelArt' / 'PA001.png',
        STYLE_WORKSPACE / 'style' / 'PixelArt' / 'PA006.png',
        STYLE_WORKSPACE / 'style' / 'PixelArt' / 'PA012.png',
    ],
    'anime_style': [
        STYLE_WORKSPACE / 'style' / 'AnimeStyle' / 'AS001.png',
        STYLE_WORKSPACE / 'style' / 'AnimeStyle' / 'AS005.png',
        STYLE_WORKSPACE / 'style' / 'AnimeStyle' / 'AS007.png',
    ],
    'cyberpunk': [
        STYLE_WORKSPACE / 'style' / 'Cyberpunk' / 'Cyberpunk001.png',
        STYLE_WORKSPACE / 'style' / 'Cyberpunk' / 'Cyberpunk004.png',
        STYLE_WORKSPACE / 'style' / 'Cyberpunk' / 'Cyberpunk009.png',
    ],
}

for style_id, reference_paths in STYLE_REFERENCE_FILES.items():
    for reference_path in reference_paths:
        assert reference_path.exists(), reference_path


def cached_fhat_from_image_path(image_path):
    image_path = Path(image_path)
    cache_path = FEATURE_CACHE_DIR / (slugify(str(image_path.relative_to(OUTPUT_DIR))) + '.pt')
    if cache_path.exists():
        cached = torch.load(cache_path, map_location='cpu', weights_only=True)
        return [feature.to(device=device, dtype=torch.float32) for feature in cached]

    image_01 = image_path_to_01(image_path)
    features = image_to_fhat_scales(image_01)
    torch.save([feature.detach().cpu() for feature in features], cache_path)
    return features


def normalized_flatten(feature):
    return F.normalize(feature.detach().float().flatten(), dim=0)


def mean_pairwise_cosine(vectors, max_pairs=MAX_PAIRWISE_COSINE_PAIRS):
    if len(vectors) < 2:
        return float('nan')
    stack = F.normalize(torch.stack(vectors, dim=0), dim=1)
    total_pairs = stack.shape[0] * (stack.shape[0] - 1) // 2
    if total_pairs <= max_pairs:
        sim = stack @ stack.T
        upper = torch.triu(torch.ones_like(sim, dtype=torch.bool), diagonal=1)
        return sim[upper].mean().item()

    rng = random.Random(0)
    values = []
    for _ in range(max_pairs):
        i = rng.randrange(stack.shape[0])
        j = rng.randrange(stack.shape[0] - 1)
        if j >= i:
            j += 1
        values.append(F.cosine_similarity(stack[i], stack[j], dim=0).item())
    return float(np.mean(values))


def stack_mean_direction(vectors):
    return F.normalize(torch.stack(vectors, dim=0).mean(dim=0), dim=0)


def reference_phi_vectors_by_style():
    reference_vectors = {scale_id: {} for scale_id in range(len(PATCH_NUMS))}
    total = sum(len(paths) for paths in STYLE_REFERENCE_FILES.values())
    progress = tqdm(total=total, desc='Reference style features')
    for style_id, reference_paths in STYLE_REFERENCE_FILES.items():
        for reference_path in reference_paths:
            reference_features = cached_fhat_from_image_path(reference_path)
            for scale_id, feature in enumerate(reference_features):
                signature = svd_signature(feature, rank=FEATURE_SWEEP_RANK, alpha=FEATURE_SWEEP_ALPHA)
                reference_vectors[scale_id].setdefault(style_id, []).append(normalized_flatten(signature['phi']))
            progress.update(1)
    progress.close()

    reference_prototypes = {scale_id: {} for scale_id in range(len(PATCH_NUMS))}
    for scale_id in range(len(PATCH_NUMS)):
        for style_id, vectors in reference_vectors[scale_id].items():
            reference_prototypes[scale_id][style_id] = stack_mean_direction(vectors)
    return reference_vectors, reference_prototypes


def plot_heatmap(table, value_column, title, output_path, vmin=None, vmax=None):
    pivot = table.pivot(index='style_id', columns='scale_id', values=value_column)
    plt.figure(figsize=(12, 4.8))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='coolwarm', center=0, vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel('VAR scale')
    plt.ylabel('style')
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()


if RUN_ALL_SCALE_STYLE_DIRECTION_SWEEP:
    direction_vectors = {scale_id: {style_id: [] for style_id in FEATURE_SWEEP_STYLE_IDS} for scale_id in range(len(PATCH_NUMS))}
    sample_rows = []
    total = len(FEATURE_SWEEP_OBJECT_IDS) * len(FEATURE_SWEEP_STYLE_IDS) * len(FEATURE_SWEEP_SEEDS)
    progress = tqdm(total=total, desc='Text-style directions across objects/seeds')

    for object_id in FEATURE_SWEEP_OBJECT_IDS:
        for seed in FEATURE_SWEEP_SEEDS:
            photo_path = STYLE_TEXT_DIR / object_id / f'photo_seed_{seed:02d}.png'
            photo_features = cached_fhat_from_image_path(photo_path)
            for style_id in FEATURE_SWEEP_STYLE_IDS:
                style_path = STYLE_TEXT_DIR / object_id / f'{style_id}_seed_{seed:02d}.png'
                style_features = cached_fhat_from_image_path(style_path)
                for scale_id, (photo_feature, style_feature) in enumerate(zip(photo_features, style_features)):
                    raw_delta = style_feature - photo_feature
                    normalized_delta = normalized_flatten(raw_delta).detach().cpu()
                    direction_vectors[scale_id][style_id].append(normalized_delta)
                    sample_rows.append({
                        'object_id': object_id,
                        'style_id': style_id,
                        'seed': seed,
                        'scale_id': scale_id,
                        'resolution': f'{PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}',
                        'relative_delta_norm': raw_delta.norm().item() / photo_feature.norm().clamp_min(1e-8).item(),
                    })
                progress.update(1)
            del photo_features
            torch.cuda.empty_cache()
    progress.close()

    sample_table = pd.DataFrame(sample_rows)
    sample_table.to_csv(FEATURE_SWEEP_DIR / 'per_sample_delta_norms.csv', index=False)

    style_prototypes = {scale_id: {} for scale_id in range(len(PATCH_NUMS))}
    consistency_rows = []
    for scale_id in range(len(PATCH_NUMS)):
        for style_id in FEATURE_SWEEP_STYLE_IDS:
            vectors = [vector.to(device) for vector in direction_vectors[scale_id][style_id]]
            prototype = stack_mean_direction(vectors).detach().cpu()
            style_prototypes[scale_id][style_id] = prototype
            within = mean_pairwise_cosine([vector.cpu() for vector in vectors])
            other_style_cosines = []
            for other_style_id in FEATURE_SWEEP_STYLE_IDS:
                if other_style_id == style_id:
                    continue
                other_vectors = [vector.to(device) for vector in direction_vectors[scale_id][other_style_id]]
                other_prototype = stack_mean_direction(other_vectors)
                other_style_cosines.append(cosine(prototype.to(device), other_prototype))
            between = float(np.mean(other_style_cosines))
            consistency_rows.append({
                'style_id': style_id,
                'scale_id': scale_id,
                'resolution': f'{PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}',
                'within_style_pairwise_cosine': within,
                'between_style_prototype_cosine': between,
                'style_consistency_gap': within - between,
                'num_vectors': len(vectors),
            })

    consistency_table = pd.DataFrame(consistency_rows)
    consistency_table.to_csv(FEATURE_SWEEP_DIR / 'style_direction_consistency_by_scale.csv', index=False)
    display(consistency_table.round(4))

    reference_vectors, reference_prototypes = reference_phi_vectors_by_style()
    alignment_rows = []
    for scale_id in range(len(PATCH_NUMS)):
        for text_style_id in FEATURE_SWEEP_STYLE_IDS:
            text_prototype = style_prototypes[scale_id][text_style_id].to(device)
            for reference_style_id in FEATURE_SWEEP_STYLE_IDS:
                reference_prototype = reference_prototypes[scale_id][reference_style_id].to(device)
                alignment_rows.append({
                    'scale_id': scale_id,
                    'resolution': f'{PATCH_NUMS[scale_id]}x{PATCH_NUMS[scale_id]}',
                    'text_style_id': text_style_id,
                    'reference_style_id': reference_style_id,
                    'is_diagonal_match': text_style_id == reference_style_id,
                    'prototype_cosine': cosine(text_prototype, reference_prototype),
                })
    alignment_table = pd.DataFrame(alignment_rows)
    alignment_table.to_csv(FEATURE_SWEEP_DIR / 'text_direction_to_reference_phi_alignment.csv', index=False)

    gap_rows = []
    for scale_id, group in alignment_table.groupby('scale_id'):
        diagonal = group[group.is_diagonal_match].prototype_cosine.mean()
        off_diagonal = group[~group.is_diagonal_match].prototype_cosine.mean()
        gap_rows.append({
            'scale_id': scale_id,
            'resolution': f'{PATCH_NUMS[int(scale_id)]}x{PATCH_NUMS[int(scale_id)]}',
            'diagonal_reference_alignment': diagonal,
            'off_diagonal_reference_alignment': off_diagonal,
            'reference_diagonal_gap': diagonal - off_diagonal,
        })
    gap_table = pd.DataFrame(gap_rows)
    gap_table.to_csv(FEATURE_SWEEP_DIR / 'reference_alignment_diagonal_gap_by_scale.csv', index=False)
    display(gap_table.round(4))

    plot_heatmap(
        consistency_table,
        'style_consistency_gap',
        'Text style direction consistency: within-style minus between-style, by scale',
        FEATURE_SWEEP_DIR / 'heatmap_style_consistency_gap.png',
    )

    plot_heatmap(
        sample_table.groupby(['style_id', 'scale_id'], as_index=False).relative_delta_norm.mean(),
        'relative_delta_norm',
        'How strongly each style phrase changes VAE feature magnitude, by scale',
        FEATURE_SWEEP_DIR / 'heatmap_relative_delta_norm.png',
        vmin=0,
    )

    best_scale = int(gap_table.sort_values('reference_diagonal_gap', ascending=False).iloc[0].scale_id)
    matrix = alignment_table[alignment_table.scale_id == best_scale].pivot(
        index='text_style_id', columns='reference_style_id', values='prototype_cosine'
    )
    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0)
    plt.title(f'Text direction vs reference Phi alignment at best scale {best_scale}')
    plt.xlabel('reference style image category')
    plt.ylabel('text style direction')
    plt.tight_layout()
    plt.savefig(FEATURE_SWEEP_DIR / f'best_scale_{best_scale}_reference_alignment_matrix.png', dpi=180, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.plot(gap_table.scale_id, gap_table.reference_diagonal_gap, marker='o', label='reference diagonal gap')
    plt.axhline(0, color='black', linewidth=1)
    plt.xticks(range(len(PATCH_NUMS)), [f'{i}\n{p}x{p}' for i, p in enumerate(PATCH_NUMS)])
    plt.ylabel('diagonal - off-diagonal cosine')
    plt.xlabel('VAR scale')
    plt.title('Does reference-image style align with text-style directions?')
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(FEATURE_SWEEP_DIR / 'reference_diagonal_gap_by_scale.png', dpi=180, bbox_inches='tight')
    plt.show()

    print('Saved all-scale sweep diagnostics in:', FEATURE_SWEEP_DIR)


## 4. Evidence-Driven Research Survey: Establish the Representation Before Designing a Method

After Section 3.2, the next step is not another injection trick. The next step is to read the statistics: which scales produce consistent text-style directions, whether those directions separate different styles, and whether reference-image Phi features align with the matching text-style direction.

This section is the decision framework for interpreting the sweep. Each study has a falsifiable hypothesis, a measurable result, and a clear stop/go decision.

### Study A: Is text style represented consistently across objects?

Generate a controlled factorial set with matched seeds:

```text
10 held-out objects x 5 text styles x 4 seeds
styles: photo, pencil sketch, watercolor, oil painting, pixel art
```

At every generation scale `l`, compute the normalized change caused by style text:

```text
d[o, style, seed, l] = normalize(F[o, style, seed, l] - F[o, photo, seed, l])
```

Then compare two quantities:

```text
within_style[l]  = mean cosine(d[o, style, l], d[o', style, l])
between_style[l] = mean cosine(d[o, style, l], d[o, other_style, l])
style_consistency[l] = within_style[l] - between_style[l]
```

| Result | Interpretation | Decision |
| --- | --- | --- |
| `style_consistency[l] > 0` at one or more scales | a style phrase creates a partly object-independent direction | continue to causal intervention |
| no positive separation | the apparent style change is object/seed-specific | do not use additive feature directions |

This is more important than visual anecdotes. It tests the central assumption behind all feature-direction methods.

### Study B: Does a reference image live near the corresponding text-style direction?

For several references from each StyleGallery style category, extract a VAE principal feature:

```text
S_ref[reference, l] = Phi(F_vae(reference, l))
```

For each text style, form an AR-direction prototype from Study A:

```text
D_text[style, l] = mean over objects and seeds of d[o, style, seed, l]
```

Build a cross-style alignment matrix at every scale:

```text
M[l, text_style, reference_style] = cosine(D_text[text_style, l], S_ref[reference_style, l])
```

The important test is not the absolute cosine. It is **diagonal dominance**:

```text
diagonal_gap[l] = mean diagonal(M[l]) - mean off-diagonal(M[l])
```

| Result | Interpretation | Decision |
| --- | --- | --- |
| positive diagonal gap | image references contain a style signal that is aligned enough to calibrate | investigate a bridge from reference to AR direction |
| flat or negative diagonal gap | raw VAE and AR features are not aligned | never inject raw VAE features directly into AR generation |

Use category labels only for this analysis. At inference, the style image must stand on its own.

### Study C: Causal transfer on held-out objects

Only run this if Study A finds a consistent AR scale. Hold out several object classes from prototype construction, then test whether a style prototype changes their rendering without destroying their identity.

```text
target feature <- target feature + alpha * D_text[style, l]
```

Sweep:

```text
scale l        : every VAR scale
strength alpha : 0.1, 0.25, 0.5, 1.0
style          : sketch, watercolor, oil painting, pixel art
```

Evaluate each result with three separate criteria:

| Criterion | What it answers |
| --- | --- |
| Object fidelity | does the target object remain recognizable? |
| Style recognizability | do blinded viewers identify the intended rendering category? |
| Image quality | is the output coherent enough to be a valid comparison? |

CLIP and SVD scores may support the analysis, but human side-by-side ratings are necessary because neither metric cleanly separates style from subject content.

### Study D: Choose the reference-image bridge only after A-C

There are two scientifically distinct outcomes.

**Outcome 1: text directions transfer, but image-reference alignment is weak.**

This means VAR-CLIP understands style only in its text-conditioning space. The honest contribution is a **reference-to-style-category calibration** study: use CLIP image-to-text similarity to retrieve a distribution over style descriptors, then test whether the corresponding AR prototype mixture transfers. This is training-free, but it controls category-level style rather than personal style.

**Outcome 2: text directions transfer and reference alignment is measurable.**

Then a compact learned bridge becomes justified:

```text
frozen CLIP image embedding
-> small adapter
-> weights over existing AR style prototypes
-> weighted AR style direction
```

The adapter predicts only prototype weights; VAR-CLIP remains frozen. Train it with style-category supervision and held-out-reference evaluation. This is far more constrained and interpretable than learning an arbitrary image-to-transformer feature mapper.

### The research gap this survey can support

A credible paper direction is not “apply PFB to VAR.” It is:

> **Characterize whether visual autoregressive models contain scale-specific, object-transferable style directions, measure their alignment with reference-image representations, and establish when reference-driven style control is possible without retraining the generator.**

That contribution has a valid negative result as well: if the surveys fail, they reveal that the feature geometry required by diffusion/Infinity-style interventions is absent in VAR-CLIP. That is useful evidence, not a failed implementation.
